# RTE: test-day inference on Colab (fallback)

Use this only if running on the laptop fails. Runtime → Change runtime type → **T4 GPU** (CPU also works, just slower).

You need three files: the code zip (`rte_code_submission.zip`), the models zip (`rte_models.zip`) and the test file.
Either upload them when asked (cell 1), or put them in a Google Drive folder and set `DRIVE_DIR` below.


In [ ]:
import os, glob
DRIVE_DIR = ""  # e.g. "/content/drive/MyDrive/rte_handoff" if the zips are in your Drive; leave "" to upload

%cd /content
if DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    for f in glob.glob(os.path.join(DRIVE_DIR, "*")):
        !cp "{f}" /content/
else:
    from google.colab import files
    print("Select rte_code_submission.zip, rte_models.zip and the test file (all at once)")
    files.upload()
!ls -la /content

## 2. Unpack code and models, install (keeps Colab's own PyTorch)

In [ ]:
%cd /content
!unzip -q -o rte_code_submission.zip -d /content
!unzip -q -o rte_models.zip -d /content/rte-challenge
%cd /content/rte-challenge
!grep -vE '^(torch|nvidia-|triton|cuda-)' requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt
!pip install -q --no-deps -e .
!ls models

## 3. Predict with all three models
Set `TEST` to the test file's name.

In [ ]:
TEST = "/content/test.jsonl"   # <- change to the downloaded test file (.jsonl / .json / .csv / .tsv)
!python scripts/predict_test.py --input "$TEST" --out predictions
!ls -la predictions

## 4. Download the predictions

In [ ]:
!zip -q -r predictions.zip predictions
from google.colab import files
files.download("predictions.zip")